Skincare Product Recommendation System
📌 Overview
This project is an AI-based recommendation system that suggests skincare products based on:
🧴 Skin Type
💡 Skin Concern
💰 Budget (in THB)
The system uses:
TF-IDF to convert ingredients into numerical vectors
Cosine Similarity to measure similarity between products and user needs

⚙️ Features
✅ Interactive dropdown (no typing required)
✅ Budget input in Thai Baht (THB)
✅ Recommends Top 10 products
✅ Displays price in THB

In [9]:
!pip install pandas scikit-learn

In [10]:
import pandas as pd

# โหลดไฟล์จาก VS Code (ต้องอยู่ path เดียวกัน)
df = pd.read_csv('product_info.csv')

df.head()

,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,P473671,Fragrance Discovery Set,6342,19-69,6320,3.6364,11.0,NaN,NaN,NaN,...,1,0,0,"['Unisex/ Genderless Scent', 'Warm &Spicy Scen...",Fragrance,Value & Gift Sets,Perfume Gift Sets,0,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342,19-69,3827,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,85.0,30.0
2,P473662,Rainbow Bar Eau de Parfum,6342,19-69,3253,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
3,P473660,Kasbah Eau de Parfum,6342,19-69,3018,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0
4,P473658,Purple Haze Eau de Parfum,6342,19-69,2691,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,1,0,0,"['Unisex/ Genderless Scent', 'Layerable Scent'...",Fragrance,Women,Perfume,2,75.0,30.0


In [11]:
df.columns = df.columns.str.lower().str.strip()
print(df.columns)

Index(['product_id', 'product_name', 'brand_id', 'brand_name', 'loves_count',
       'rating', 'reviews', 'size', 'variation_type', 'variation_value',
       'variation_desc', 'ingredients', 'price_usd', 'value_price_usd',
       'sale_price_usd', 'limited_edition', 'new', 'online_only',
       'out_of_stock', 'sephora_exclusive', 'highlights', 'primary_category',
       'secondary_category', 'tertiary_category', 'child_count',
       'child_max_price', 'child_min_price'],
      dtype='object')


In [12]:
df = df[['product_name', 'brand_name', 'ingredients', 'price_usd', 'highlights']]

In [13]:
df = df.dropna(subset=['ingredients', 'price_usd'])

In [14]:
def map_skin_type(text):
    text = str(text).lower()
    
    if 'oily' in text:
        return 'oily'
    elif 'dry' in text:
        return 'dry'
    elif 'sensitive' in text:
        return 'sensitive'
    else:
        return 'normal'

df['skin_type'] = df['highlights'].apply(map_skin_type)

df['skin_type'].value_counts()

skin_type
normal    5519
dry       1423
oily       607
Name: count, dtype: int64

In [15]:
def recommend(df, skin_type, concern, budget):
    
    # 🔥 แปลงบาท → USD
    budget = budget / 35
    
    df_filtered = df[df['price_usd'] <= budget].copy()
    df_filtered = df_filtered[df_filtered['skin_type'] == skin_type]
    
    if df_filtered.shape[0] == 0:
        print("❌ No products found")
        return
    
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(df_filtered['ingredients'])
    
    user_vec = tfidf.transform([concern])
    similarity = cosine_similarity(user_vec, tfidf_matrix)
    
    df_filtered['score'] = similarity.flatten()
    df_sorted = df_filtered.sort_values(by='score', ascending=False)
    
    top10 = df_sorted.head(10).copy()
    top10['price_thb'] = top10['price_usd'] * 35
    
    return top10[['product_name', 'brand_name', 'price_thb', 'score']]
      


    

In [16]:
import ipywidgets as widgets
from IPython.display import display

# Dropdown เลือกประเภทผิว
skin_widget = widgets.Dropdown(
    options=['oily', 'dry', 'sensitive', 'normal'],
    description='Skin Type:',
    style={'description_width': 'initial'}
)

# Dropdown เลือกปัญหาผิว
concern_widget = widgets.Dropdown(
    options=['acne', 'dark spots', 'brightening', 'anti-aging'],
    description='Skin Concern:',
    style={'description_width': 'initial'}
)

# Dropdown เลือกงบ
budget_widget = widgets.Dropdown(
    options=[300, 500, 1000, 1500],
    description='Budget (THB):',
    style={'description_width': 'initial'}
)

# ปุ่มกด
button = widgets.Button(description="✨ Recommend Products")

# output แสดงผล
output = widgets.Output()

# ฟังก์ชันตอนกดปุ่ม
def on_button_click(b):
    with output:
        output.clear_output()
        
        result = recommend(
            df,
            skin_widget.value,
            concern_widget.value,
            budget_widget.value
        )
        
        display(result)

# ผูกปุ่มกับฟังก์ชัน
button.on_click(on_button_click)

# แสดงทั้งหมด
display(skin_widget, concern_widget, budget_widget, button, output)

Dropdown(description='Skin Type:', options=('oily', 'dry', 'sensitive', 'normal'), style=DescriptionStyle(desc…

Dropdown(description='Skin Concern:', options=('acne', 'dark spots', 'brightening', 'anti-aging'), style=Descr…

Dropdown(description='Budget (THB):', options=(300, 500, 1000, 1500), style=DescriptionStyle(description_width…

Button(description='✨ Recommend Products', style=ButtonStyle())

Output()